# EDA — wypowiedzi Sejmu, IX kadencja (2019–2023)

Cel: sprawdzić jakość `dataset.csv`, rozkład klubów i długości tekstów oraz odfiltrować wypowiedzi zbyt krótkie / proceduralne.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

MIN_CHARS = 300  # próg długości pod trening klasyfikatora

data_dir = Path("../../data/term9")
dataset_path = data_dir / "dataset.csv"
clean_path = data_dir / "dataset_clean.csv"

df = pd.read_csv(dataset_path)
df["text_len"] = df["text"].str.len()
df["date"] = pd.to_datetime(df["date"])

print(f"Wiersze: {len(df):,}")
print(f"Posłowie (member_id): {df['member_id'].nunique()}")
print(f"Zakres dat: {df['date'].min().date()} – {df['date'].max().date()}")
print(f"Kolumny: {list(df.columns)}")
df.head()

## Rozkład klubów

In [ ]:
club_counts = df["club"].value_counts()
club_share = (club_counts / len(df) * 100).round(1)

club_summary = pd.DataFrame({"count": club_counts, "share_%": club_share})
display(club_summary)

fig, ax = plt.subplots(figsize=(9, 4))
club_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Liczba wypowiedzi wg klubu")
ax.set_xlabel("Klub")
ax.set_ylabel("Liczba wypowiedzi")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Długość tekstu

In [ ]:
display(df["text_len"].describe().to_frame("text_len"))

thresholds = [50, 100, 200, 300, 500, 1000]
rows = []
for t in thresholds:
    n = int((df["text_len"] < t).sum())
    rows.append({"threshold": t, "below": n, "share_%": round(n / len(df) * 100, 2)})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["text_len"], bins=50, color="steelblue")
axes[0].set_title("Długość wypowiedzi (wszystkie)")
axes[0].set_xlabel("Znaki")
axes[0].set_ylabel("Liczba")

capped = df.loc[df["text_len"] <= 5000, "text_len"]
axes[1].hist(capped, bins=50, color="steelblue")
axes[1].axvline(MIN_CHARS, color="crimson", linestyle="--", label=f"próg {MIN_CHARS}")
axes[1].set_title("Długość wypowiedzi (≤ 5000 znaków)")
axes[1].set_xlabel("Znaki")
axes[1].set_ylabel("Liczba")
axes[1].legend()

plt.tight_layout()
plt.show()

## Krótkie / proceduralne wypowiedzi

Bardzo krótkie teksty to często mikroreakcje (`Nie.`, `Serio?`), fragmenty ślubowania albo komunikaty organizacyjne — słaby sygnał ideologiczny.

In [ ]:
short = df[df["text_len"] < MIN_CHARS].sort_values("text_len")
print(f"Wypowiedzi < {MIN_CHARS} znaków: {len(short):,} ({len(short) / len(df) * 100:.2f}%)")
display(short[["date", "speaker_name", "club", "text_len", "text"]].head(20))

oath = df["text"].str.contains(r"Ślubuję", case=False, na=False)
print(f"Teksty zawierające 'Ślubuję': {oath.sum()}")

## Filtrowanie i zapis `dataset_clean.csv`

Reguła: zostawiamy wypowiedzi o długości ≥ `MIN_CHARS` (domyślnie 300) oraz bez ślubowania.

In [ ]:
oath_mask = df["text"].str.contains(r"Ślubuję", case=False, na=False)
keep = (df["text_len"] >= MIN_CHARS) & ~oath_mask

df_clean = df.loc[keep].drop(columns=["text_len"]).copy()
dropped = len(df) - len(df_clean)

print(f"Przed: {len(df):,}")
print(f"Po:    {len(df_clean):,}")
print(f"Usunięto: {dropped:,} ({dropped / len(df) * 100:.2f}%)")
print()
print("Rozkład klubów po filtrze:")
display(
    pd.DataFrame({
        "count": df_clean["club"].value_counts(),
        "share_%": (df_clean["club"].value_counts(normalize=True) * 100).round(1),
    })
)

df_clean.to_csv(clean_path, index=False)
print(f"Zapisano: {clean_path.resolve()}")

## Wnioski

- Dataset ma ~36.6k wypowiedzi, 10 klubów; dominują **KO** i **PiS** (silny imbalance).
- Mediana długości ~1100 znaków; bardzo krótkich tekstów jest mało (<300 znaków ≈ 1%).
- Do treningu używamy `dataset_clean.csv` (próg długości + bez ślubowań).
- Następny krok: mapowanie `club` → `side` (prawica/lewica) i split po `member_id`.